In [1]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate

In [2]:
ds_toxigen = load_dataset(
    "toxigen/toxigen-data",
    split="train"
)

def preprocess_toxigen(example):
    label = 1 if example["toxicity_human"] >= 3 else 0
    return {"text": example["text"], "label": label}

ds_toxigen = ds_toxigen.map(
    preprocess_toxigen,
    remove_columns=ds_toxigen.column_names
)

print(ds_toxigen)

Dataset({
    features: ['text', 'label'],
    num_rows: 8960
})


In [3]:
ds_multi_en = load_dataset(
    "textdetox/multilingual_toxicity_dataset",
    split="en"
)

ds_multi_ru = load_dataset(
    "textdetox/multilingual_toxicity_dataset",
    split="ru"
)

def preprocess_multi(example):
    return {"text": example["text"], "label": example["toxic"]}

ds_multi_en = ds_multi_en.map(
    preprocess_multi,
    remove_columns=ds_multi_en.column_names
)

ds_multi_ru = ds_multi_ru.map(
    preprocess_multi,
    remove_columns=ds_multi_ru.column_names
)

ds_multi = concatenate_datasets([ds_multi_en, ds_multi_ru])

print(ds_multi)

Dataset({
    features: ['text', 'label'],
    num_rows: 10000
})


In [4]:
ds_ru1 = load_dataset(
    "Xeonil/ru-merged-toxic-comments",
    split="train"
)

def preprocess_ru1(example):
    return {"text": example["text"], "label": example["target"]}

ds_ru1 = ds_ru1.map(
    preprocess_ru1,
    remove_columns=ds_ru1.column_names
)

print(ds_ru1)

Dataset({
    features: ['text', 'label'],
    num_rows: 271410
})


In [5]:
ds_ru2 = load_dataset(
    "AlexSham/Toxic_Russian_Comments",
    split="train"
)

def preprocess_ru2(example):
    return {"text": example["text"], "label": example["label"]}

ds_ru2 = ds_ru2.map(
    preprocess_ru2,
    remove_columns=ds_ru2.column_names
)

print(ds_ru2)

Dataset({
    features: ['text', 'label'],
    num_rows: 223461
})


In [6]:
full_dataset = concatenate_datasets([
    ds_toxigen,
    ds_multi,
    ds_ru1,
    ds_ru2
])

full_dataset = full_dataset.shuffle(seed=42)
full_dataset = full_dataset.train_test_split(test_size=0.1)

print(full_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 462447
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 51384
    })
})


In [7]:
train_df = full_dataset['train'].to_pandas()
test_df = full_dataset['test'].to_pandas()

df = pd.concat([train_df, test_df], axis=0)

# Text Preprocessing

In [8]:
import re
import spacy

nlp_ru = spacy.load("ru_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def lemmatize_text(text):
    if re.search(r'[а-яА-Я]', text):
        doc = nlp_ru(text)
    else:
        doc = nlp_en(text)
        
    lemmas = [token.lemma_ for token in doc if not token.is_stop]
    return " ".join(lemmas)

In [9]:
df['text_clean'] = df['text'].apply(clean_text)
df['text_lemma'] = df['text_clean'].apply(lemmatize_text)

train_df['text_clean'] = train_df['text'].apply(clean_text)
train_df['text_lemma'] = train_df['text_clean'].apply(lemmatize_text)

test_df['text_clean'] = test_df['text'].apply(clean_text)
test_df['text_lemma'] = test_df['text_clean'].apply(lemmatize_text)

In [10]:
df.to_csv('data.csv')
train_df.to_csv('train_data.csv')
test_df.to_csv('test_data.csv')